In [1]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Config
# ============================================================
CHUNK_CSV = "data_files/chunked_paragraphs.csv"
GITHUB_URL = "https://raw.githubusercontent.com/US-CBO/eval-projections/main/output_data/outlay_projection_errors.csv"

OUTPUT_PARQUET = "data_files/flattened_cbo_reports_synonym_match.parquet"
SYNONYM_JSON = "data_files/subcategory_synonyms_word2vec.json"

# Generate N nearest-neighbor "synonyms" per label using Word2Vec similar_by_vector
N_SYNONYMS_PER_LABEL = 20

# If True: regenerate synonyms from Word2Vec and OVERWRITE the JSON file
REGENERATE_SYNONYMS = True

# If True: even when loading an existing JSON, rewrite it after applying hygiene cleanup
ALWAYS_REWRITE_JSON_AFTER_CLEANING = True

# Generic junk / dangerous tokens that create false matches
FORBIDDEN_EXACT = {
    "security", "benefits", "insurance", "health",
    "spending", "outlays", "program", "programs",
    "mandatory", "discretionary", "total",
    "net", "interest", "other", "some", "certain", "various", "most", "several", "multiple"
}
FORBIDDEN_SUBSTRINGS = {
    "list", "alternative", "newline", "return", "exactly", "abbreviations"
}

# Extra “wrong security” guardrails (drop neighbors clearly about homeland/transport security)
SECURITY_CONTEXT_FORBIDDEN = {
    "homeland", "aviation", "transportation", "tsa", "terror", "terrorism", "border"
}

# ============================================================
# Load paragraphs
# ============================================================
df2 = pd.read_csv(CHUNK_CSV).reset_index(drop=True)
df2["row_id"] = np.arange(len(df2), dtype=np.int64)

for col in ["component", "category", "subcategory", "match_method"]:
    if col not in df2.columns:
        df2[col] = pd.NA

# ============================================================
# Load official CBO mapping + official subcategory list
# ============================================================
df_errors = pd.read_csv(GITHUB_URL)
mapping = (
    df_errors[["subcategory", "category", "component"]]
    .dropna(subset=["subcategory", "category", "component"])
    .drop_duplicates()
    .drop_duplicates(subset=["subcategory"], keep="first")
)
subcategories = sorted(mapping["subcategory"].unique().tolist())
print(f"Loaded {len(subcategories)} official subcategories:\n{subcategories}")

# ============================================================
# Text normalization
#   (1) makes Medi_Cal == Medi-Cal == Medi Cal
#   (2) makes matching ignore '_' and '-' differences
# ============================================================
def normalize_text(x):
    x = "" if pd.isna(x) else str(x).lower()

    # Treat underscores and hyphens as whitespace separators
    x = re.sub(r"[_\-]+", " ", x)

    # Optionally soften other punctuation to spaces (keeps letters/numbers)
    x = re.sub(r"[^\w\s]", " ", x)

    # Collapse whitespace
    x = re.sub(r"\s+", " ", x).strip()
    return x

# ============================================================
# Hygiene filter for formatting/artifact tokens like:
#   -----_Net, =====_Net, ----------------_Total, FINANCIAL_DETAILS, etc.
# ============================================================
def is_formatting_artifact_token(w: str) -> bool:
    if not w:
        return True

    # lots of dashes/equals/underscores anywhere
    if re.search(r"[-=]{6,}", w):
        return True

    # starts with separator runs
    if re.match(r"^[-=_]{4,}", w):
        return True

    # tokens that are mostly punctuation/underscore
    letters = len(re.findall(r"[A-Za-z]", w))
    digits = len(re.findall(r"\d", w))
    punct  = len(w) - letters - digits
    if letters == 0 and digits == 0:
        return True
    if punct / max(len(w), 1) > 0.55:
        return True

    # repeated underscores (common in scraped formatting)
    if re.search(r"_{3,}", w):
        return True

    return False

# ============================================================
# Word2Vec synonym generation (vector -> nearest words)
# ============================================================
syn_path = Path(SYNONYM_JSON)

def keep_candidate_token(label: str, w: str) -> bool:
    wl = w.lower()
    label_l = label.lower()

    # reject formatting/artifact tokens
    if is_formatting_artifact_token(w):
        return False

    # reject prompt-echo / instruction artifacts
    if any(bad in wl for bad in FORBIDDEN_SUBSTRINGS):
        return False

    # reject exact generic junk
    if wl in FORBIDDEN_EXACT:
        return False

    # reject numeric / weird tokens
    if re.fullmatch(r"\d+", w):
        return False
    if len(w) <= 2:
        return False
    if re.search(r"\d", w):
        return False

    # prevent overlap across program labels (avoid "Medicaid" showing up as a Medicare synonym, etc.)
    for other in subcategories:
        if other.lower() == label_l:
            continue
        other_norm = normalize_text(other).replace(" ", "_")  # normalized form for containment checks
        if other_norm and other_norm in wl:
            return False

    # Social Security: eliminate homeland/transport security drift and "security" alone
    if label == "Social Security":
        if wl == "security":
            return False
        if any(tok in wl for tok in SECURITY_CONTEXT_FORBIDDEN):
            return False

    return True

def label_to_vector(label: str, model):
    """
    Build a vector for a label:
    1) Try underscore phrase token (e.g., Social_Security)
    2) Else average vectors for words in the label (case-sensitive model)
    """
    phrase_token = label.replace(" ", "_")
    if phrase_token in model:
        return model[phrase_token].astype(np.float32)

    parts = label.split()
    vecs = [model[p] for p in parts if p in model]
    if not vecs:
        vecs = [model[p.title()] for p in parts if p.title() in model]
    if not vecs:
        return None
    return np.mean(np.vstack(vecs), axis=0).astype(np.float32)

if (not syn_path.exists()) or REGENERATE_SYNONYMS:
    print("\nGenerating synonyms using Word2Vec similar_by_vector (OVERWRITING JSON)...")
    import gensim.downloader as api
    w2v_model = api.load("word2vec-google-news-300")

    synonyms = {}
    debug_kept = {}

    for sub in subcategories:
        v = label_to_vector(sub, w2v_model)
        if v is None:
            synonyms[sub] = [sub]
            debug_kept[sub] = 0
            continue

        nbrs = w2v_model.similar_by_vector(v, topn=1200)

        kept = []
        for w, score in nbrs:
            # drop the label token itself
            if w.lower() in {sub.lower(), sub.replace(" ", "_").lower()}:
                continue
            if not keep_candidate_token(sub, w):
                continue
            kept.append(w)
            if len(kept) >= N_SYNONYMS_PER_LABEL:
                break

        synonyms[sub] = [sub] + kept
        debug_kept[sub] = len(kept)

    syn_path.parent.mkdir(parents=True, exist_ok=True)
    syn_path.write_text(json.dumps(synonyms, indent=2), encoding="utf-8")
    print(f"Saved synonyms to: {SYNONYM_JSON}")

    print("\nNeighbors kept per subcategory:")
    print(pd.Series(debug_kept).sort_values(ascending=False))

else:
    print(f"\nLoading synonyms from: {SYNONYM_JSON}")
    synonyms = json.loads(syn_path.read_text(encoding="utf-8"))

    # Apply hygiene cleanup even when loading, and optionally rewrite the JSON file
    cleaned = {}
    for sub, syns in synonyms.items():
        kept = []
        for w in syns:
            # keep official label always
            if str(w).strip().lower() == sub.lower():
                kept.append(sub)
                continue
            if keep_candidate_token(sub, str(w)):
                kept.append(str(w))
        # de-dupe preserve order
        seen = set()
        out = []
        for w in kept:
            k = w.lower()
            if k in seen:
                continue
            seen.add(k)
            out.append(w)
        cleaned[sub] = out

    synonyms = cleaned

    if ALWAYS_REWRITE_JSON_AFTER_CLEANING:
        syn_path.write_text(json.dumps(synonyms, indent=2), encoding="utf-8")
        print(f"Rewrote cleaned synonyms to: {SYNONYM_JSON}")

# ============================================================
# Build phrase match patterns (official + Word2Vec neighbors)
#   - uses normalize_text() so Medi_Cal matches Medi-Cal, etc.
# ============================================================
phrase_records = []
for sub, syns in synonyms.items():
    for phrase in syns:
        pnorm = normalize_text(phrase)
        if not pnorm:
            continue
        if pnorm in FORBIDDEN_EXACT:
            continue
        # also drop phrases that normalize into formatting artifacts
        if is_formatting_artifact_token(phrase):
            continue
        phrase_records.append((phrase, sub))

# longest-first so more specific phrases win
phrase_records = sorted(phrase_records, key=lambda x: len(str(x[0])), reverse=True)

compiled = []
for phrase, sub in phrase_records:
    pnorm = normalize_text(phrase)
    pat = re.compile(r"(?<!\w)" + re.escape(pnorm) + r"(?!\w)", flags=re.IGNORECASE)
    compiled.append((pat, sub, phrase))

def match_by_phrases(text: str):
    t = normalize_text(text)
    for pat, sub, phrase in compiled:
        if pat.search(t):
            return sub, phrase
    return pd.NA, pd.NA

# ============================================================
# Apply matching (direct only)
# ============================================================
print("\nMatching paragraphs by direct phrase hits (official + Word2Vec neighbors)...")

matched_subs = []
matched_phrase = []

for txt in df2["text"].astype(str):
    sub, phrase = match_by_phrases(txt)
    matched_subs.append(sub)
    matched_phrase.append(phrase)

df2["subcategory"] = matched_subs
df2["matched_phrase"] = matched_phrase
df2["match_method"] = np.where(df2["subcategory"].notna(), "direct_phrase_w2v_neighbors", pd.NA)

# attach category/component
df2 = df2.drop(columns=["category", "component"], errors="ignore").merge(mapping, on="subcategory", how="left")

print("\nMatch counts:")
print(df2["match_method"].value_counts(dropna=False))
print("\nTop subcategories:")
print(df2["subcategory"].value_counts(dropna=False).head(15))

# ============================================================
# Save parquet (no embeddings)
# ============================================================
print(f"\nWriting parquet: {OUTPUT_PARQUET}")
df2.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow", compression="zstd")
print("Done.")

Loaded 10 official subcategories:
['Defense Discretionary', 'Medicaid', 'Medicare', 'Net Interest', 'Nondefense Discretionary', 'Other Mandatory', 'Social Security', 'Total', 'Total Discretionary', 'Total Mandatory']

Generating synonyms using Word2Vec similar_by_vector (OVERWRITING JSON)...
Saved synonyms to: data_files/subcategory_synonyms_word2vec.json

Neighbors kept per subcategory:
Defense Discretionary       20
Medicaid                    20
Medicare                    20
Net Interest                20
Nondefense Discretionary    20
Other Mandatory             20
Social Security             20
Total                       20
Total Discretionary         20
Total Mandatory             20
dtype: int64

Matching paragraphs by direct phrase hits (official + Word2Vec neighbors)...

Match counts:
match_method
direct_phrase_w2v_neighbors    22010
<NA>                           12784
Name: count, dtype: int64

Top subcategories:
subcategory
<NA>                        12784
Total Mandator